# Enhanced LLM-Aided Testbench Generation — Homework

**Course:** LLM4ChipDesign · **Student:** Anuj Apte (NYU Tandon, MS Computer Engineering) · **Assignment:** `Enhanced_Testbench_Generation_Assignment.pdf`

This notebook runs the course's 5-step pipeline (description + RTL → initial testbench → Python golden model → patterns with golden outputs → self-checking final testbench) and verifies everything with Icarus Verilog.

| Part | What | Where in this notebook |
|---|---|---|
| I | Two tutorial examples end-to-end: **2-to-1 mux** (combinational) and **4-bit adder** (arithmetic) | Sections 3–4 |
| II | Inspect and explain golden model, pattern file, updater effect (4-bit adder) | Section 5 |
| III | Own module: **`sat_adder8`** — 8-bit unsigned saturating adder with carry-in | Section 6 |
| IV | Bug detection: inject bug → failing run → fix → passing run (+ mutation sweep) | Section 7 |

Artifacts for each run are written to `mux/`, `adder/`, `custom/` (`testbench_initial.v`, `golden_model.py`, `test_patterns_with_golden.json`, `testbench_final.v`, plus the DUT and simulation logs).

### How the LLM is provided (read this)
The pipeline classes below are the course's own (`LLMClient`, `TestbenchGenerator`, `GoldenModelGenerator`, `TestbenchUpdater`, `TestbenchPipeline`), with three small, marked edits to `LLMClient` that add a **`replay` provider**.

* **Live mode** — put an OpenAI key in the one obvious cell in Section 1 (or export `OPENAI_API_KEY`). The pipeline then calls `gpt-4o` exactly as in the tutorial notebook.
* **Replay mode (key left empty — how the outputs below were produced)** — each LLM call is answered from `llm_responses/<module>/…`, a recorded response that **Claude (Anthropic) wrote** for that exact pipeline prompt, because the environment this run was prepared in had no OpenAI access. Everything downstream of the LLM — parsing, running the golden model, writing artifacts, `iverilog`/`vvp` — is real, live execution. The report states this plainly.

## 1. Setup and API key

In [1]:
import os, sys, json, re, shutil, subprocess, difflib, importlib.util, platform
from typing import Dict, Any, List, Optional

# --- dependencies (quiet no-ops if already present) ---------------------------
if importlib.util.find_spec("openai") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=False)
if shutil.which("iverilog") is None and platform.system() == "Linux":
    pre = [] if os.geteuid() == 0 else ["sudo"]
    subprocess.run(pre + ["apt-get", "install", "-y", "-q", "iverilog"], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Python    :", sys.version.split()[0])
print("iverilog  :", subprocess.run("iverilog -V", shell=True, capture_output=True, text=True).stdout.splitlines()[0]
      if shutil.which("iverilog") else "NOT FOUND - install Icarus Verilog")
print("Working dir:", os.getcwd())

Python    : 3.13.16
iverilog  : Icarus Verilog version 12.0 (stable) ()
Working dir: /home/claude/work/HW_TestbenchGen


In [2]:
# >>>>>>>>>>  INSERT YOUR OPENAI API KEY HERE (leave empty for replay mode)  <<<<<<<<<<
OPENAI_API_KEY = ""
# Never commit a real key. Alternatively export OPENAI_API_KEY before starting Jupyter.

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if os.environ.get("OPENAI_API_KEY"):
    LLM_MODE, LLM_MODEL = "openai", "gpt-4o"
    print("Live mode: calling OpenAI model", LLM_MODEL)
else:
    LLM_MODE, LLM_MODEL = "replay", "claude-sonnet-5-5 (recorded responses in llm_responses/)"
    print("Replay mode: LLM responses come from llm_responses/ (authored by Claude).")
    print("             Insert an OpenAI key above to run the same notebook live.")

Replay mode: LLM responses come from llm_responses/ (authored by Claude).
             Insert an OpenAI key above to run the same notebook live.


## 2. Pipeline implementation (course code)

The five cells below are the course notebook's classes. Edits relative to the tutorial notebook (all in `LLMClient`):
1. `openai` is imported optionally (only live mode needs it); a stray unused `xml.parsers.expat` import was removed.
2. The OpenAI client is only constructed when `provider == "openai"`.
3. A `replay` provider returns the recorded response matching the pipeline stage and module name.

`TestbenchGenerator`, `GoldenModelGenerator`, `TestbenchUpdater` and `TestbenchPipeline` are unmodified, including all prompts.

### 2.1 `LLMClient`

In [3]:
"""
LLM Client for interacting with language models.
Supports multiple LLM providers (OpenAI, Anthropic, etc.)
"""

import os
import json
from typing import Optional, Dict, Any
try:
    import openai
except ImportError:      # only needed in live OpenAI mode
    openai = None


class LLMClient:
    """Client for interacting with LLM APIs."""

    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4", provider: str = "openai"):
        """
        Initialize LLM client.

        Args:
            api_key: API key for the LLM provider (if None, reads from environment)
            model: Model name to use
            provider: LLM provider ('openai', 'anthropic', etc.)
        """
        self.provider = provider
        self.model = model
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")
        # --- edit 2: only build an OpenAI client in live mode ---
        self.client = openai.OpenAI(api_key=self.api_key) if provider == "openai" else None

    def generate(self, prompt: str, system_prompt: Optional[str] = None,
                 temperature: float = 0.7, max_tokens: int = 4000) -> str:
        """
        Generate text using the LLM.

        Args:
            prompt: User prompt
            system_prompt: System prompt for the model
            temperature: Sampling temperature (0-1)
            max_tokens: Maximum tokens to generate

        Returns:
            Generated text response
        """
        try:
            if self.provider == "openai":
                messages = []
                if system_prompt:
                    messages.append({"role": "system", "content": system_prompt})
                messages.append({"role": "user", "content": prompt})

                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                return response.choices[0].message.content
            elif self.provider == "replay":
                # --- edit 3: replay a recorded response (see llm_responses/) ---
                return self._replay(prompt, system_prompt)
            else:
                raise ValueError(f"Unsupported provider: {self.provider}")
        except Exception as e:
            print(f"Error generating response: {e}")
            return f"Error: {str(e)}"

    # ------------------------------------------------------------------
    # Replay provider: serves the recorded LLM response for (stage, module).
    # Stage is recognised from the pipeline's own system prompt.
    # ------------------------------------------------------------------
    REPLAY_DIR = "llm_responses"
    _STAGES = [("expert in Verilog testbench generation", "01_testbench_response.txt"),
               ("expert in hardware design and Python", "02_golden_response.txt"),
               ("expert in Verilog testbench development", "03_updater_response.txt")]

    def _replay(self, prompt: str, system_prompt: Optional[str]) -> str:
        import re
        stage_file = next((f for key, f in self._STAGES if key in (system_prompt or "")), None)
        m = re.search(r"Module Name:\s*(\w+)", prompt)
        if not m:   # stage 1: take the module name from the Verilog code section
            code = prompt.split("Verilog Module Code:")[-1]
            m = re.search(r"\bmodule\s+(\w+)\s*\(", code)
        if not stage_file or not m:
            raise RuntimeError("replay: could not identify pipeline stage / module name")
        path = os.path.join(self.REPLAY_DIR, m.group(1), stage_file)
        with open(path, encoding="utf-8") as f:
            text = f.read()
        print(f"    [replay] {path}  ({len(text):,} chars)")
        return text

    def is_available(self) -> bool:
        """Check if the LLM client is properly configured."""
        return self.api_key is not None and len(self.api_key) > 0

### 2.2 `TestbenchGenerator` (Step 3)

In [4]:
"""
Step 3: Generate testbench with test patterns (without golden outputs).
"""

from typing import Dict, Any, List


class TestbenchGenerator:
    """Generate Verilog testbench with test patterns using LLM."""

    def __init__(self, llm_client: LLMClient):
        """
        Initialize testbench generator.

        Args:
            llm_client: LLM client instance
        """
        self.llm_client = llm_client

    def generate_testbench(self, description: str, verilog_code: str) -> Dict[str, Any]:
        """
        Generate testbench with comprehensive test patterns.

        Args:
            description: Natural language description of the Verilog module
            verilog_code: Verilog code to be tested

        Returns:
            Dictionary containing:
                - testbench_code: Verilog testbench code (without expected outputs)
                - test_patterns: List of test input patterns
                - module_info: Information about module (name, inputs, outputs)
        """
        # First, extract module information
        module_info = self._extract_module_info(verilog_code)

        # Generate comprehensive test patterns
        system_prompt = """You are an expert in Verilog testbench generation.
Your task is to generate comprehensive test patterns for a given Verilog module.
Generate test patterns that cover:
1. All corner cases
2. Boundary values
3. Typical use cases
4. Edge cases
5. Random values for thorough testing"""

        user_prompt = f"""Given the following Verilog module and its natural language description,
generate a comprehensive Verilog testbench that includes ALL possible test patterns.

Natural Language Description:
{description}

Verilog Module Code:
{verilog_code}

Generate a Verilog testbench that:
1. Declares all necessary signals
2. Instantiates the module under test
3. Includes a systematic set of test patterns covering all cases
4. Uses $display to show inputs for each test (all $display statements are in the initial block and before $finish statement.)
5. Does NOT include expected outputs or assertions yet (we will add those later)
6. Numbers each test case

Please provide:
1. The complete testbench code
2. A list of test patterns in JSON format with test number and input values

Format your response as:
TESTBENCH_CODE:
```verilog
[testbench code here]
```

TEST_PATTERNS (a list of dictionaries. Each dictionary contains only input signal names mapped to their values as plain binary strings (no prefixes like 0b, no spaces). Do not include any test_number field):
```json
[array of test patterns]
```
"""

        response = self.llm_client.generate(user_prompt, system_prompt, max_tokens=4000)

        # Parse the response
        testbench_code = self._extract_section(response, "TESTBENCH_CODE:", "```verilog", "```")
        test_patterns_json = self._extract_section(response, "TEST_PATTERNS:", "```json", "```")

        try:
            test_patterns = eval(test_patterns_json) if test_patterns_json else []
        except:
            test_patterns = []
            print("Warning: Could not parse test patterns JSON")

        return {
            "testbench_code": testbench_code,
            "test_patterns": test_patterns,
            "module_info": module_info,
            "raw_response": response
        }

    def _extract_module_info(self, verilog_code: str) -> Dict[str, Any]:
        """
        Extract module information (name, inputs, outputs) from Verilog code.

        Args:
            verilog_code: Verilog module code

        Returns:
            Dictionary with module information
        """
        lines = verilog_code.strip().split('\n')
        module_name = ""
        inputs = []
        outputs = []

        for line in lines:
            line = line.strip()
            if line.startswith('module'):
                # Extract module name
                parts = line.split()
                if len(parts) >= 2:
                    module_name = parts[1].split('(')[0]
            elif 'input' in line:
                # Extract input signals
                input_part = line.replace('input', '').replace(';', '').replace(',', '').strip()
                if input_part:
                    inputs.append(input_part)
            elif 'output' in line:
                # Extract output signals
                output_part = line.replace('output', '').replace(';', '').replace(',', '').strip()
                if output_part:
                    outputs.append(output_part)

        return {
            "module_name": module_name,
            "inputs": inputs,
            "outputs": outputs
        }

    def _extract_section(self, text: str, marker: str, start_delim: str, end_delim: str) -> str:
        """
        Extract a section from the LLM response between delimiters.

        Args:
            text: Full response text
            marker: Section marker to find
            start_delim: Start delimiter (e.g., "```verilog")
            end_delim: End delimiter (e.g., "```")

        Returns:
            Extracted section content
        """
        try:
            # Find the marker
            marker_idx = text.find(marker)
            if marker_idx == -1:
                return ""

            # Find the start delimiter after the marker
            start_idx = text.find(start_delim, marker_idx)
            if start_idx == -1:
                return ""
            start_idx += len(start_delim)

            # Find the end delimiter
            end_idx = text.find(end_delim, start_idx)
            if end_idx == -1:
                return ""

            return text[start_idx:end_idx].strip()
        except Exception as e:
            print(f"Error extracting section: {e}")
            return ""

### 2.3 `GoldenModelGenerator` (Step 4)

In [5]:
"""
Step 4: Generate Python golden model and compute golden outputs.
"""

import sys
import io
import json
from typing import Dict, Any, List


class GoldenModelGenerator:
    """Generate Python golden model and compute expected outputs."""

    def __init__(self, llm_client: LLMClient):
        """
        Initialize golden model generator.

        Args:
            llm_client: LLM client instance
        """
        self.llm_client = llm_client

    def generate_python_model(self, description: str, module_info: Dict[str, Any]) -> str:
        """
        Generate Python implementation based on natural language description.

        Args:
            description: Natural language description of the module
            module_info: Module information (name, inputs, outputs)

        Returns:
            Python code implementing the module functionality
        """
        system_prompt = """You are an expert in hardware design and Python programming.
Your task is to create a Python function that implements the exact same functionality
as described in the natural language specification."""

        user_prompt = f"""Given the following natural language description of a hardware module,
create a Python function that implements this functionality.

Natural Language Description:
{description}

Module Information:
- Module Name: {module_info.get('module_name', 'unknown')}
- Inputs: {module_info.get('inputs', [])}
- Outputs: {module_info.get('outputs', [])}

Create a Python function named '{module_info.get('module_name', 'module')}_golden' that:
1. Takes the input signals as parameters
2. Computes and returns the output signals
3. Implements the exact functionality described
4. Handles all edge cases properly
5. Returns outputs as a dictionary with output signal names as keys

Provide ONLY the Python function code, no explanations.
Start with 'def {module_info.get('module_name', 'module')}_golden(' and include complete implementation.
"""

        response = self.llm_client.generate(user_prompt, system_prompt, temperature=0.2, max_tokens=3000)

        # Extract Python code
        python_code = self._extract_python_code(response)

        return python_code

    def compute_golden_outputs(self, python_code: str, test_patterns: List[Dict[str, Any]],
                               module_info: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Execute the Python golden model with test patterns to get expected outputs.

        Args:
            python_code: Python golden model code
            test_patterns: List of test input patterns
            module_info: Module information

        Returns:
            List of test patterns with golden outputs added
        """
        results = []

        # Execute the Python code in a safe namespace
        namespace = {}
        try:
            exec(python_code, namespace)
        except Exception as e:
            print(f"Error executing Python code: {e}")
            return results

        # Find the golden function
        function_name = f"{module_info.get('module_name', 'module')}_golden"
        golden_func = namespace.get(function_name)

        if not golden_func:
            print(f"Error: Could not find function {function_name}")
            return results

        # Run each test pattern through the golden model
        for pattern in test_patterns:
            try:
                # Extract input values from the pattern
                inputs = pattern.get('inputs', pattern)

                # Call the golden function
                if isinstance(inputs, dict):
                    outputs = golden_func(**inputs)
                else:
                    # If inputs is not a dict, try to call with positional args
                    outputs = golden_func(*inputs.values()) if hasattr(inputs, 'values') else golden_func(inputs)

                # Add outputs to the pattern
                result = pattern.copy()
                result['expected_outputs'] = outputs
                results.append(result)
            except Exception as e:
                print(f"Error computing golden output for pattern {pattern}: {e}")
                result = pattern.copy()
                result['expected_outputs'] = None
                result['error'] = str(e)
                results.append(result)

        return results

    def _extract_python_code(self, text: str) -> str:
        """
        Extract Python code from LLM response.

        Args:
            text: LLM response text

        Returns:
            Extracted Python code
        """
        # Try to find code between ```python and ```
        if "```python" in text:
            start = text.find("```python") + len("```python")
            end = text.find("```", start)
            if end != -1:
                return text[start:end].strip()

        # Try to find code between ``` and ```
        if "```" in text:
            parts = text.split("```")
            if len(parts) >= 3:
                return parts[1].strip()

        # If no code blocks found, look for def statement
        if "def " in text:
            lines = text.split('\n')
            code_lines = []
            in_function = False
            for line in lines:
                if line.strip().startswith('def '):
                    in_function = True
                if in_function:
                    code_lines.append(line)
            return '\n'.join(code_lines)

        return text.strip()

### 2.4 `TestbenchUpdater` (Step 5)

In [6]:
"""
Step 5: Update testbench with golden outputs.
"""

from typing import Dict, Any, List
import re
import json


class TestbenchUpdater:
    """Update generated testbench with golden outputs using LLM."""

    def __init__(self, llm_client: LLMClient):
        """
        Initialize testbench updater.

        Args:
            llm_client: LLM client instance
        """
        self.llm_client = llm_client

    def update_testbench(self, testbench_code: str, test_patterns_with_outputs: List[Dict[str, Any]],
                        module_info: Dict[str, Any]) -> str:
        """
        Update testbench code to include expected outputs and verification.

        Args:
            testbench_code: Original testbench code without expected outputs
            test_patterns_with_outputs: Test patterns with golden outputs
            module_info: Module information

        Returns:
            Updated testbench code with assertions and expected outputs
        """
        # Use LLM if available, otherwise fall back to rule-based approach
        if self.llm_client.is_available():
            updated_code = self._llm_update_testbench(testbench_code, test_patterns_with_outputs, module_info)
        else:
            # Fallback to rule-based approach
            updated_code = self._add_verification_logic(testbench_code, test_patterns_with_outputs, module_info)

        return updated_code

    def _llm_update_testbench(self, testbench_code: str, test_patterns_with_outputs: List[Dict[str, Any]],
                             module_info: Dict[str, Any]) -> str:
        """
        Use LLM to update testbench with verification logic.

        Args:
            testbench_code: Original testbench code
            test_patterns_with_outputs: Test patterns with golden outputs
            module_info: Module information

        Returns:
            Updated testbench code with verification logic
        """
        system_prompt = """You are an expert in Verilog testbench development and verification.
Your task is to update a testbench by adding comprehensive verification logic and expected output checks.
You should:
1. Add verification for each test case using the provided expected outputs
2. Track passed and failed test counts
3. Display clear pass/fail messages for each output signal
4. Generate a comprehensive test summary at the end
5. Maintain the original testbench structure and style
6. Use proper Verilog syntax and best practices"""

        # Prepare test patterns data for the LLM
        patterns_str = json.dumps(test_patterns_with_outputs, indent=2)

        user_prompt = f"""Given the following Verilog testbench and test patterns with expected outputs,
update the testbench to include verification logic that checks the actual outputs against the expected outputs.

Original Testbench Code:
```verilog
{testbench_code}
```

Module Information:
- Module Name: {module_info.get('module_name', 'unknown')}
- Inputs: {module_info.get('inputs', [])}
- Outputs: {module_info.get('outputs', [])}

Test Patterns with Expected Outputs:
```json
{patterns_str}
```

Please update the testbench to:
1. Add integer variables 'passed_tests' and 'failed_tests' at the beginning of the initial block (initialized to 0)
2. After each test case (identified by $display statements), add a delay (#10) for outputs to settle
3. For each output signal, compare the actual value against the expected value from the test patterns
4. Display "✓" for passing checks and "✗" for failing checks with actual and expected values
5. Increment passed_tests for each passing check and failed_tests for each failing check
6. At the end of the initial block (before 'end'), add a test summary showing:
   - Total tests run
   - Number passed
   - Number failed
7. Preserve all original test case displays and structure
8. Use proper indentation and formatting

Provide ONLY the complete updated testbench code, no explanations.
Format your response as:
```verilog
[updated testbench code here]
```
"""

        response = self.llm_client.generate(user_prompt, system_prompt, max_tokens=8000)

        # Extract the Verilog code from the response
        updated_code = self._extract_verilog_code(response)

        # If extraction failed, fall back to original with rule-based update
        if not updated_code or len(updated_code) < len(testbench_code) // 2:
            print("Warning: LLM response extraction failed, using rule-based approach")
            updated_code = self._add_verification_logic(testbench_code, test_patterns_with_outputs, module_info)

        return updated_code

    def _extract_verilog_code(self, text: str) -> str:
        """
        Extract Verilog code from LLM response.

        Args:
            text: LLM response text

        Returns:
            Extracted Verilog code
        """
        # Try to find code between ```verilog and ```
        if "```verilog" in text:
            start = text.find("```verilog") + len("```verilog")
            end = text.find("```", start)
            if end != -1:
                return text[start:end].strip()

        # Try to find code between ``` and ```
        if "```" in text:
            parts = text.split("```")
            if len(parts) >= 3:
                # Get the first code block
                code = parts[1].strip()
                # If it starts with a language identifier, remove it
                if code.startswith("verilog\n") or code.startswith("verilog "):
                    code = code.split('\n', 1)[1] if '\n' in code else code
                return code.strip()

        # If no code blocks found, look for module or testbench keywords
        if "module " in text or "initial " in text:
            return text.strip()

        return ""

    def _add_verification_logic(self, testbench_code: str, test_patterns: List[Dict[str, Any]],
                               module_info: Dict[str, Any]) -> str:
        """
        Add verification logic to the testbench.

        Args:
            testbench_code: Original testbench code
            test_patterns: Test patterns with expected outputs
            module_info: Module information

        Returns:
            Testbench code with verification logic added
        """
        lines = testbench_code.split('\n')
        updated_lines = []

        # Track if we're in the initial block
        in_initial = False
        indent_level = 0
        test_case_num = 0

        for i, line in enumerate(lines):
            stripped = line.strip()

            # Detect initial block
            if 'initial' in stripped and 'begin' in stripped:
                in_initial = True
                updated_lines.append(line)
                # Add test result tracking variables after initial begin
                updated_lines.append("    integer passed_tests = 0;")
                updated_lines.append("    integer failed_tests = 0;")
                updated_lines.append("")
                continue

            # Check for test case markers (e.g., $display for test cases)
            if in_initial and '$display' in stripped and ('Test' in stripped or 'test' in stripped):
                # This is likely a test case display
                updated_lines.append(line)

                # Add delay to let outputs settle
                indent = len(line) - len(line.lstrip())
                indent_str = ' ' * indent
                updated_lines.append(f"{indent_str}#10; // Wait for outputs to settle")

                # Add verification for this test case if we have expected outputs
                if test_case_num < len(test_patterns):
                    pattern = test_patterns[test_case_num]
                    if 'expected_outputs' in pattern and pattern['expected_outputs']:
                        verification_lines = self._generate_verification(
                            pattern, module_info, indent
                        )
                        updated_lines.extend(verification_lines)
                    test_case_num += 1
                continue

            # Check for end of initial block
            if in_initial and (stripped == 'end' or stripped.startswith('end')):
                # Add final summary before the end
                indent = len(line) - len(line.lstrip())
                indent_str = ' ' * indent
                updated_lines.append("")
                updated_lines.append(f"{indent_str}// Test Summary")
                updated_lines.append(f'{indent_str}$display("\\n========== Test Summary ==========");')
                updated_lines.append(f'{indent_str}$display("Total Tests: %0d", passed_tests + failed_tests);')
                updated_lines.append(f'{indent_str}$display("Passed: %0d", passed_tests);')
                updated_lines.append(f'{indent_str}$display("Failed: %0d", failed_tests);')
                updated_lines.append(f'{indent_str}$display("==================================\\n");')
                updated_lines.append("")
                updated_lines.append(line)
                in_initial = False
                continue

            updated_lines.append(line)

        return '\n'.join(updated_lines)

    def _generate_verification(self, pattern: Dict[str, Any], module_info: Dict[str, Any],
                              indent: int) -> List[str]:
        """
        Generate verification code for a single test pattern.

        Args:
            pattern: Test pattern with expected outputs
            expected_outputs: Expected output values
            module_info: Module information
            indent: Indentation level

        Returns:
            List of verification code lines
        """
        lines = []
        indent_str = ' ' * indent

        expected = pattern.get('expected_outputs', {})
        if not expected:
            return lines

        # Get output signals
        outputs = module_info.get('outputs', [])

        # Generate verification for each output
        for output in outputs:
            output_name = output.split('[')[0].strip()  # Remove bit width if present
            output_name = output_name.split()[-1]  # Get the signal name

            if output_name in expected:
                expected_value = expected[output_name]

                # Check if expected value is boolean
                if isinstance(expected_value, bool):
                    expected_value = 1 if expected_value else 0

                # Generate comparison
                lines.append(f"{indent_str}if ({output_name} === {expected_value}) begin")
                lines.append(f'{indent_str}    $display("  ✓ {output_name} = %b (expected: {expected_value})", {output_name});')
                lines.append(f"{indent_str}    passed_tests = passed_tests + 1;")
                lines.append(f"{indent_str}end else begin")
                lines.append(f'{indent_str}    $display("  ✗ {output_name} = %b (expected: {expected_value})", {output_name});')
                lines.append(f"{indent_str}    failed_tests = failed_tests + 1;")
                lines.append(f"{indent_str}end")

        return lines

### 2.5 `TestbenchPipeline`

In [7]:
"""
Main pipeline orchestrating the entire testbench generation process.
"""

import json
import os
from typing import Dict, Any, Optional
import subprocess

class TestbenchPipeline:
    """
    Main pipeline for LLM-aided testbench generation.

    Steps:
    1. Accept natural language description and Verilog code
    2. Generate testbench with test patterns (no golden outputs)
    3. Generate Python golden model from description
    4. Execute golden model with test patterns to get expected outputs
    5. Update testbench with golden outputs and verification logic
    """

    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4", provider: str = "openai"):
        """
        Initialize the pipeline.

        Args:
            api_key: API key for LLM provider
            model: Model name to use
            provider: LLM provider name
        """
        self.llm_client = LLMClient(api_key, model, provider)
        self.testbench_gen = TestbenchGenerator(self.llm_client)
        self.golden_gen = GoldenModelGenerator(self.llm_client)
        self.testbench_updater = TestbenchUpdater(self.llm_client)

    def run(self, description: str, verilog_code: str, output_dir: str = "output") -> Dict[str, Any]:
        """
        Run the complete testbench generation pipeline.

        Args:
            description: Natural language description of the module
            verilog_code: Verilog code to be tested
            output_dir: Directory to save output files

        Returns:
            Dictionary containing all generated artifacts
        """
        print("=" * 80)
        print("LLM-Aided Testbench Generation Pipeline")
        print("=" * 80)

        # Create output directory
        os.makedirs(output_dir, exist_ok=True)

        # Step 1 & 2: Input handling (description and verilog code are already provided)
        print("\n[Step 1-2] Input: Natural language description and Verilog code received")
        print(f"Description length: {len(description)} characters")
        print(f"Verilog code length: {len(verilog_code)} characters")

        # Step 3: Generate testbench with test patterns
        print("\n[Step 3] Generating testbench with test patterns...")
        if not self.llm_client.is_available():
            print("WARNING: LLM client not configured. Using mock generation.")
            testbench_result = self._mock_testbench_generation(verilog_code)
        else:
            testbench_result = self.testbench_gen.generate_testbench(description, verilog_code)

        print(f"  - Generated testbench with {len(testbench_result['test_patterns'])} test patterns")
        print(f"  - Module: {testbench_result['module_info']['module_name']}")

        # Save initial testbench (without golden outputs)
        initial_tb_path = os.path.join(output_dir, "testbench_initial.v")
        with open(initial_tb_path, 'w') as f:
            f.write(testbench_result['testbench_code'])
        print(f"  - Saved initial testbench to: {initial_tb_path}")

        # Step 4: Generate Python golden model and compute golden outputs
        print("\n[Step 4] Generating Python golden model and computing expected outputs...")
        if not self.llm_client.is_available():
            print("WARNING: LLM client not configured. Using mock golden model.")
            python_code = self._mock_python_model(testbench_result['module_info'])
        else:
            python_code = self.golden_gen.generate_python_model(
                description,
                testbench_result['module_info']
            )

        print(f"  - Generated Python golden model ({len(python_code)} characters)")

        # change testbench_result['test_patterns'] value to integral values
        patterns = []
        for pattern in testbench_result['test_patterns']:
            for key in pattern:
                if isinstance(pattern[key], str):
                    pattern[key] = int(pattern[key], 2)
            patterns.append(pattern)
        testbench_result['test_patterns'] = patterns

        # Save Python golden model
        python_path = os.path.join(output_dir, "golden_model.py")
        with open(python_path, 'w') as f:
            f.write(python_code)
        print(f"  - Saved Python golden model to: {python_path}")

        # Compute golden outputs
        print("  - Computing golden outputs for all test patterns...")
        test_patterns_with_outputs = self.golden_gen.compute_golden_outputs(
            python_code,
            testbench_result['test_patterns'],
            testbench_result['module_info']
        )

        successful_patterns = sum(1 for p in test_patterns_with_outputs
                                 if 'expected_outputs' in p and p['expected_outputs'] is not None)
        print(f"  - Successfully computed outputs for {successful_patterns}/{len(test_patterns_with_outputs)} patterns")

        # Save test patterns with golden outputs
        patterns_path = os.path.join(output_dir, "test_patterns_with_golden.json")
        with open(patterns_path, 'w') as f:
            json.dump(test_patterns_with_outputs, f, indent=2)
        print(f"  - Saved test patterns with golden outputs to: {patterns_path}")

        # Step 5: Update testbench with golden outputs
        print("\n[Step 5] Updating testbench with golden outputs and verification logic...")
        final_testbench = self.testbench_updater.update_testbench(
            testbench_result['testbench_code'],
            test_patterns_with_outputs,
            testbench_result['module_info']
        )

        # Save final testbench
        final_tb_path = os.path.join(output_dir, "testbench_final.v")
        with open(final_tb_path, 'w') as f:
            f.write(final_testbench)
        print(f"  - Saved final testbench to: {final_tb_path}")

        print("\n" + "=" * 80)
        print("Pipeline completed successfully!")
        print("=" * 80)
        print(f"\nGenerated files in '{output_dir}':")
        print(f"  - testbench_initial.v    : Initial testbench with test patterns")
        print(f"  - golden_model.py        : Python reference implementation")
        print(f"  - test_patterns_with_golden.json : Test patterns with expected outputs")
        print(f"  - testbench_final.v      : Final testbench with verification")
        print()

        return {
            'description': description,
            'verilog_code': verilog_code,
            'module_info': testbench_result['module_info'],
            'test_patterns': testbench_result['test_patterns'],
            'initial_testbench': testbench_result['testbench_code'],
            'python_golden_model': python_code,
            'test_patterns_with_outputs': test_patterns_with_outputs,
            'final_testbench': final_testbench,
            'output_dir': output_dir
        }

    def _mock_testbench_generation(self, verilog_code: str) -> Dict[str, Any]:
        """Mock testbench generation when LLM is not available."""
        return {
            'testbench_code': '// Mock testbench - LLM not configured\n' + verilog_code,
            'test_patterns': [],
            'module_info': {
                'module_name': 'unknown',
                'inputs': [],
                'outputs': []
            }
        }

    def _mock_python_model(self, module_info: Dict[str, Any]) -> str:
        """Mock Python model generation when LLM is not available."""
        return f"# Mock Python model - LLM not configured\ndef {module_info['module_name']}_golden():\n    pass\n"

### 2.6 Helpers for simulation and inspection
`run_sim` prints the **exact** `iverilog`/`vvp` commands, shows the output, parses the pass/fail summary and saves a log next to the artifacts.

In [8]:
def make_pipeline():
    return TestbenchPipeline(api_key=os.environ.get("OPENAI_API_KEY") or "replay",
                             model="gpt-4o" if LLM_MODE == "openai" else "claude-sonnet-5-5-recorded",
                             provider=LLM_MODE)

def write(path, text):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def run_sim(dut, tb, workdir, tag="sim", log="simulation_output.log", show=True, tail=None):
    """Compile + simulate with Icarus Verilog. Returns dict(passed, failed, stdout, ok)."""
    vvp = f"{workdir}/{tag}.vvp"
    cmd_compile = f"iverilog -g2012 -o {vvp} {dut} {tb}"
    cmd_run = f"vvp {vvp}"
    lines = [f"$ {cmd_compile}"]
    c = subprocess.run(cmd_compile, shell=True, capture_output=True, text=True, encoding="utf-8", timeout=60)
    lines.append(c.stderr.strip() or "(compiled with no errors or warnings)")
    out = ""
    if c.returncode == 0:
        lines.append(f"$ {cmd_run}")
        r = subprocess.run(cmd_run, shell=True, capture_output=True, text=True, encoding="utf-8", timeout=60)
        out = r.stdout
        lines.append(out.rstrip())
    text = "\n".join(l for l in lines if l is not None)
    if log:
        write(f"{workdir}/{log}", text + "\n")
    if show:
        shown = text
        if tail:   # keep command lines, trim the long middle of the vvp output
            ol = out.rstrip().splitlines()
            shown = "\n".join(lines[:3] + ["   ... (%d lines trimmed; full log saved) ..." % max(0, len(ol) - tail)] + ol[-tail:])
        print(shown)
    p = re.search(r"Passed:\s*(\d+)", out); f = re.search(r"Failed:\s*(\d+)", out)
    return dict(passed=int(p.group(1)) if p else None, failed=int(f.group(1)) if f else None,
                stdout=out, ok=(c.returncode == 0))

def show_file(path, first=None, numbered=True):
    ls = open(path, encoding="utf-8").read().splitlines()
    ls = ls[:first] if first else ls
    print(f"--- {path} ({len(ls)} lines shown) " + "-" * 30)
    for i, l in enumerate(ls, 1):
        print(f"{i:4d}: {l}" if numbered else l)

def show_checking_sections(tb_path, tests_shown=2):
    """Print the self-checking parts of testbench_final.v: counters, first checks, summary."""
    ls = open(tb_path, encoding="utf-8").read().splitlines()
    starts = [i for i, l in enumerate(ls) if re.match(r"\s*// Test \d+", l)]
    summ = next((i for i, l in enumerate(ls) if re.search(r"test summary", l, re.I)), None)
    if len(starts) <= tests_shown or summ is None:
        show_file(tb_path); return
    def dump(a, b):
        for i in range(a, b): print(f"{i+1:4d}: {ls[i]}")
    print(f"--- {tb_path}: declarations + counters + first {tests_shown} tests ---")
    dump(0, starts[tests_shown])
    print(f"      ... ({len(starts) - tests_shown} more test blocks, same structure) ...")
    print("--- summary / pass-fail verdict ---")
    dump(summ, len(ls))

def diff_initial_final(outdir, context=1, max_lines=60):
    a = open(f"{outdir}/testbench_initial.v", encoding="utf-8").read().splitlines()
    b = open(f"{outdir}/testbench_final.v", encoding="utf-8").read().splitlines()
    d = list(difflib.unified_diff(a, b, "testbench_initial.v", "testbench_final.v", lineterm="", n=context))
    print("\n".join(d[:max_lines]))
    if len(d) > max_lines: print(f"... ({len(d) - max_lines} more diff lines)")
    added = sum(1 for l in d if l.startswith("+") and not l.startswith("+++"))
    removed = sum(1 for l in d if l.startswith("-") and not l.startswith("---"))
    print(f"\nTotals: initial = {len(a)} lines, final = {len(b)} lines  (+{added} / -{removed} in unified diff)")

def list_artifacts(d):
    for n in ["testbench_initial.v", "golden_model.py", "test_patterns_with_golden.json", "testbench_final.v"]:
        p = f"{d}/{n}"
        print(f"  {'OK ' if os.path.exists(p) else 'MISSING'} {p:48s} {os.path.getsize(p) if os.path.exists(p) else 0:>7,} bytes")
print("helpers ready")

helpers ready


## 3. Part I — Example 1: 2-to-1 multiplexer (combinational)

**DUT interface:** `mux2to1(input a, input b, input sel, output y)` — all 1-bit.

In [9]:
mux_description = """
A 2-to-1 multiplexer (MUX).

The module takes two 1-bit input signals (a and b) and one 1-bit select signal (sel).

Functionality:
- Input 'a': First data input (1-bit)
- Input 'b': Second data input (1-bit)
- Input 'sel': Select signal (1-bit)
- Output 'y': Selected output (1-bit)

When sel is 0, the output y should be equal to input a.
When sel is 1, the output y should be equal to input b.

This is a combinational logic circuit with no state or memory.

"""

mux_verilog_code = """
module mux2to1 (
    input wire a,
    input wire b,
    input wire sel,
    output wire y
);
    assign y = sel ? b : a;
endmodule

"""
write("mux/mux2to1.v", mux_verilog_code)
print(mux_description); print(mux_verilog_code)


A 2-to-1 multiplexer (MUX).

The module takes two 1-bit input signals (a and b) and one 1-bit select signal (sel).

Functionality:
- Input 'a': First data input (1-bit)
- Input 'b': Second data input (1-bit)
- Input 'sel': Select signal (1-bit)
- Output 'y': Selected output (1-bit)

When sel is 0, the output y should be equal to input a.
When sel is 1, the output y should be equal to input b.

This is a combinational logic circuit with no state or memory.



module mux2to1 (
    input wire a,
    input wire b,
    input wire sel,
    output wire y
);
    assign y = sel ? b : a;
endmodule




### 3.1 Run the pipeline (generates all four artifacts)

In [10]:
print("=" * 80, f"\nPipeline run: mux2to1   | LLM mode = {LLM_MODE} | model = {LLM_MODEL}\n", "=" * 80, sep="")
mux_result = make_pipeline().run(description=mux_description, verilog_code=mux_verilog_code, output_dir="mux")
list_artifacts("mux")

Pipeline run: mux2to1   | LLM mode = replay | model = claude-sonnet-5-5 (recorded responses in llm_responses/)
LLM-Aided Testbench Generation Pipeline

[Step 1-2] Input: Natural language description and Verilog code received
Description length: 462 characters
Verilog code length: 134 characters

[Step 3] Generating testbench with test patterns...
    [replay] llm_responses/mux2to1/01_testbench_response.txt  (1,841 chars)
  - Generated testbench with 8 test patterns
  - Module: mux2to1
  - Saved initial testbench to: mux/testbench_initial.v

[Step 4] Generating Python golden model and computing expected outputs...
    [replay] llm_responses/mux2to1/02_golden_response.txt  (207 chars)
  - Generated Python golden model (192 characters)
  - Saved Python golden model to: mux/golden_model.py
  - Computing golden outputs for all test patterns...
  - Successfully computed outputs for 8/8 patterns
  - Saved test patterns with golden outputs to: mux/test_patterns_with_golden.json

[Step 5] Updat

### 3.2 Generated artifacts

In [11]:
show_file("mux/testbench_initial.v")

--- mux/testbench_initial.v (46 lines shown) ------------------------------
   1: `timescale 1ns/1ps
   2: 
   3: module tb_mux2to1;
   4:     reg a;
   5:     reg b;
   6:     reg sel;
   7:     wire y;
   8: 
   9:     mux2to1 uut (.a(a), .b(b), .sel(sel), .y(y));
  10: 
  11:     initial begin
  12:         // Test 1 (exhaustive)
  13:         a = 1'b0; b = 1'b0; sel = 1'b0;
  14:         $display("Test 1: a=%b b=%b sel=%b", a, b, sel);
  15: 
  16:         // Test 2 (exhaustive)
  17:         a = 1'b1; b = 1'b0; sel = 1'b0;
  18:         $display("Test 2: a=%b b=%b sel=%b", a, b, sel);
  19: 
  20:         // Test 3 (exhaustive)
  21:         a = 1'b0; b = 1'b1; sel = 1'b0;
  22:         $display("Test 3: a=%b b=%b sel=%b", a, b, sel);
  23: 
  24:         // Test 4 (exhaustive)
  25:         a = 1'b1; b = 1'b1; sel = 1'b0;
  26:         $display("Test 4: a=%b b=%b sel=%b", a, b, sel);
  27: 
  28:         // Test 5 (exhaustive)
  29:         a = 1'b0; b = 1'b0; sel = 1'b1;
  30:  

In [12]:
show_file("mux/golden_model.py")

--- mux/golden_model.py (5 lines shown) ------------------------------
   1: def mux2to1_golden(a, b, sel):
   2:     """Reference model of a 2-to-1 multiplexer: y = b when sel == 1, else a."""
   3:     a, b, sel = a & 1, b & 1, sel & 1
   4:     y = b if sel else a
   5:     return {'y': y}


In [13]:
pats = json.load(open("mux/test_patterns_with_golden.json"))
print(f"mux/test_patterns_with_golden.json: {len(pats)} patterns; first 3 and last 1 shown")
print(json.dumps(pats[:3] + pats[-1:], indent=1))

mux/test_patterns_with_golden.json: 8 patterns; first 3 and last 1 shown
[
 {
  "a": 0,
  "b": 0,
  "sel": 0,
  "expected_outputs": {
   "y": 0
  }
 },
 {
  "a": 1,
  "b": 0,
  "sel": 0,
  "expected_outputs": {
   "y": 1
  }
 },
 {
  "a": 0,
  "b": 1,
  "sel": 0,
  "expected_outputs": {
   "y": 0
  }
 },
 {
  "a": 1,
  "b": 1,
  "sel": 1,
  "expected_outputs": {
   "y": 1
  }
 }
]


### 3.3 Evidence: the checking logic in `testbench_final.v`

In [14]:
show_checking_sections("mux/testbench_final.v", tests_shown=2)

--- mux/testbench_final.v: declarations + counters + first 2 tests ---
   1: `timescale 1ns/1ps
   2: 
   3: module tb_mux2to1;
   4:     reg a;
   5:     reg b;
   6:     reg sel;
   7:     wire y;
   8: 
   9:     mux2to1 uut (.a(a), .b(b), .sel(sel), .y(y));
  10: 
  11:     initial begin
  12:         integer passed_tests;
  13:         integer failed_tests;
  14:         passed_tests = 0;
  15:         failed_tests = 0;
  16: 
  17:         // Test 1 (exhaustive)
  18:         a = 1'b0; b = 1'b0; sel = 1'b0;
  19:         $display("Test 1: a=%b b=%b sel=%b", a, b, sel);
  20:         #10; // let outputs settle
  21:         if (y === 1'd0) begin
  22:             $display("  ✓ Test 1: y = %0d (expected: 0)", y);
  23:             passed_tests = passed_tests + 1;
  24:         end else begin
  25:             $display("  ✗ Test 1: y = %0d (expected: 0)", y);
  26:             failed_tests = failed_tests + 1;
  27:         end
  28: 
  29:         // Test 2 (exhaustive)
  30:       

### 3.4 Verification with Icarus Verilog (exact commands + output)

In [15]:
mux_sim = run_sim("mux/mux2to1.v", "mux/testbench_final.v", "mux")
assert mux_sim["ok"] and mux_sim["failed"] == 0, "mux testbench did not pass"
print(f"\n==> mux2to1 PASS: {mux_sim['passed']} checks passed, {mux_sim['failed']} failed")

$ iverilog -g2012 -o mux/sim.vvp mux/mux2to1.v mux/testbench_final.v
(compiled with no errors or warnings)
$ vvp mux/sim.vvp
Test 1: a=0 b=0 sel=0
  ✓ Test 1: y = 0 (expected: 0)
Test 2: a=1 b=0 sel=0
  ✓ Test 2: y = 1 (expected: 1)
Test 3: a=0 b=1 sel=0
  ✓ Test 3: y = 0 (expected: 0)
Test 4: a=1 b=1 sel=0
  ✓ Test 4: y = 1 (expected: 1)
Test 5: a=0 b=0 sel=1
  ✓ Test 5: y = 0 (expected: 0)
Test 6: a=1 b=0 sel=1
  ✓ Test 6: y = 0 (expected: 0)
Test 7: a=0 b=1 sel=1
  ✓ Test 7: y = 1 (expected: 1)
Test 8: a=1 b=1 sel=1
  ✓ Test 8: y = 1 (expected: 1)

========== Test Summary ==========
Total Checks: 8
Passed: 8
Failed: 0
ALL TESTS PASSED

mux/testbench_final.v:121: $finish called at 80000 (1ps)

==> mux2to1 PASS: 8 checks passed, 0 failed


## 4. Part I — Example 2: 4-bit adder (arithmetic)

**DUT interface:** `adder4bit(input [3:0] a, input [3:0] b, output [3:0] sum, output carry)`.

In [16]:
adder_description = """
A simple 4-bit adder module.

The module takes two 4-bit input signals (a and b) and produces a 4-bit sum output and a 1-bit carry output.

Functionality:
- Input 'a': 4-bit unsigned number
- Input 'b': 4-bit unsigned number
- Output 'sum': 4-bit result of a + b (lower 4 bits)
- Output 'carry': 1-bit carry-out flag (set to 1 if result exceeds 15)

The adder performs unsigned addition of the two 4-bit inputs.
If the result is greater than 15 (0xF), the carry output should be set to 1.

"""

adder_verilog_code = """
module adder4bit (
    input wire [3:0] a,
    input wire [3:0] b,
    output wire [3:0] sum,
    output wire carry
);
    wire [4:0] result;
    assign result = a + b;
    assign sum = result[3:0];
    assign carry = result[4];
endmodule

"""
write("adder/adder4bit.v", adder_verilog_code)
print(adder_description); print(adder_verilog_code)


A simple 4-bit adder module.

The module takes two 4-bit input signals (a and b) and produces a 4-bit sum output and a 1-bit carry output.

Functionality:
- Input 'a': 4-bit unsigned number
- Input 'b': 4-bit unsigned number
- Output 'sum': 4-bit result of a + b (lower 4 bits)
- Output 'carry': 1-bit carry-out flag (set to 1 if result exceeds 15)

The adder performs unsigned addition of the two 4-bit inputs.
If the result is greater than 15 (0xF), the carry output should be set to 1.



module adder4bit (
    input wire [3:0] a,
    input wire [3:0] b,
    output wire [3:0] sum,
    output wire carry
);
    wire [4:0] result;
    assign result = a + b;
    assign sum = result[3:0];
    assign carry = result[4];
endmodule




### 4.1 Run the pipeline

In [17]:
print("=" * 80, f"\nPipeline run: adder4bit | LLM mode = {LLM_MODE} | model = {LLM_MODEL}\n", "=" * 80, sep="")
adder_result = make_pipeline().run(description=adder_description, verilog_code=adder_verilog_code, output_dir="adder")
list_artifacts("adder")

Pipeline run: adder4bit | LLM mode = replay | model = claude-sonnet-5-5 (recorded responses in llm_responses/)
LLM-Aided Testbench Generation Pipeline

[Step 1-2] Input: Natural language description and Verilog code received
Description length: 491 characters
Verilog code length: 241 characters

[Step 3] Generating testbench with test patterns...
    [replay] llm_responses/adder4bit/01_testbench_response.txt  (5,242 chars)
  - Generated testbench with 32 test patterns
  - Module: adder4bit
  - Saved initial testbench to: adder/testbench_initial.v

[Step 4] Generating Python golden model and computing expected outputs...
    [replay] llm_responses/adder4bit/02_golden_response.txt  (294 chars)
  - Generated Python golden model (279 characters)
  - Saved Python golden model to: adder/golden_model.py
  - Computing golden outputs for all test patterns...
  - Successfully computed outputs for 32/32 patterns
  - Saved test patterns with golden outputs to: adder/test_patterns_with_golden.json


### 4.2 Evidence: the checking logic in `testbench_final.v`

In [18]:
show_checking_sections("adder/testbench_final.v", tests_shown=2)

--- adder/testbench_final.v: declarations + counters + first 2 tests ---
   1: `timescale 1ns/1ps
   2: 
   3: module tb_adder4bit;
   4:     reg [3:0] a;
   5:     reg [3:0] b;
   6:     wire [3:0] sum;
   7:     wire carry;
   8: 
   9:     adder4bit uut (.a(a), .b(b), .sum(sum), .carry(carry));
  10: 
  11:     initial begin
  12:         integer passed_tests;
  13:         integer failed_tests;
  14:         passed_tests = 0;
  15:         failed_tests = 0;
  16: 
  17:         // Test 1 (corner)
  18:         a = 4'b0000; b = 4'b0000;
  19:         $display("Test 1: a=%b b=%b", a, b);
  20:         #10; // let outputs settle
  21:         if (sum === 4'd0) begin
  22:             $display("  ✓ Test 1: sum = %0d (expected: 0)", sum);
  23:             passed_tests = passed_tests + 1;
  24:         end else begin
  25:             $display("  ✗ Test 1: sum = %0d (expected: 0)", sum);
  26:             failed_tests = failed_tests + 1;
  27:         end
  28:         if (carry === 1'd

### 4.3 Verification with Icarus Verilog (exact commands + output)

In [19]:
adder_sim = run_sim("adder/adder4bit.v", "adder/testbench_final.v", "adder", tail=14)
assert adder_sim["ok"] and adder_sim["failed"] == 0, "adder testbench did not pass"
print(f"\n==> adder4bit PASS: {adder_sim['passed']} checks passed, {adder_sim['failed']} failed")

$ iverilog -g2012 -o adder/sim.vvp adder/adder4bit.v adder/testbench_final.v
(compiled with no errors or warnings)
$ vvp adder/sim.vvp
   ... (91 lines trimmed; full log saved) ...
  ✓ Test 31: sum = 15 (expected: 15)
  ✓ Test 31: carry = 0 (expected: 0)
Test 32: a=0011 b=1001
  ✓ Test 32: sum = 12 (expected: 12)
  ✓ Test 32: carry = 0 (expected: 0)

========== Test Summary ==========
Total Checks: 64
Passed: 64
Failed: 0
ALL TESTS PASSED

adder/testbench_final.v:633: $finish called at 320000 (1ps)

==> adder4bit PASS: 64 checks passed, 0 failed


## 5. Part II — Inspecting the artifacts (4-bit adder)

Each cell below prints the facts that the explanation refers to, taken from the files just generated. (The prose describes the recorded run; if you re-run in live mode the generated code can differ in wording and layout, though the structure is the same.)

### 5.1 Golden model — `adder/golden_model.py`

In [20]:
import ast
src = open("adder/golden_model.py", encoding="utf-8").read()
print(src)
fn = next(n for n in ast.parse(src).body if isinstance(n, ast.FunctionDef))
print("function name :", fn.name)
print("parameters    :", [a.arg for a in fn.args.args])
ns = {}; exec(src, ns)
for a, b in [(3, 4), (9, 9), (15, 15)]:
    print(f"  {fn.name}({a:2d}, {b:2d}) -> {ns[fn.name](a, b)}")

def adder4bit_golden(a, b):
    """Reference model of a 4-bit unsigned adder with carry-out."""
    total = (a & 0xF) + (b & 0xF)          # full-precision unsigned sum (0..30)
    return {'sum': total & 0xF,            # lower 4 bits
            'carry': 1 if total > 15 else 0}
function name : adder4bit_golden
parameters    : ['a', 'b']
  adder4bit_golden( 3,  4) -> {'sum': 7, 'carry': 0}
  adder4bit_golden( 9,  9) -> {'sum': 2, 'carry': 1}
  adder4bit_golden(15, 15) -> {'sum': 14, 'carry': 1}


**What the golden model expects and produces**

* **Inputs:** one Python parameter per DUT input port, named identically (`a`, `b`). The pipeline calls it as `adder4bit_golden(**inputs)`, with each input already converted from the LLM's binary string to a plain `int` (`int(x, 2)`), so `a`/`b` are unsigned integers 0–15.
* **Output:** a `dict` keyed by DUT output port name (`sum`, `carry`) with integer values — the same names the testbench compares.
* **How the reference is computed:** it never looks at the Verilog. It implements the *natural-language spec*: add the operands at full precision (Python ints do not wrap), then `sum` is the low 4 bits (`& 0xF`) and `carry` is 1 exactly when the full result exceeds 15. Masking the inputs also keeps it well-defined for out-of-range values. Because it is written from the spec and the RTL is written separately, a disagreement flags a bug in one of them.

### 5.2 Pattern file — `adder/test_patterns_with_golden.json`

In [21]:
pats = json.load(open("adder/test_patterns_with_golden.json"))
print("top-level type :", type(pats).__name__, "| records:", len(pats))
print("record fields  :", list(pats[0].keys()))
print("one record     :", json.dumps(pats[0]))
print("a carry record :", json.dumps(next(p for p in pats if p["expected_outputs"]["carry"] == 1)))
print("value types    :", {k: type(v).__name__ for k, v in pats[0].items()},
      "| expected_outputs types:", {k: type(v).__name__ for k, v in pats[0]["expected_outputs"].items()})

top-level type : list | records: 32
record fields  : ['a', 'b', 'expected_outputs']
one record     : {"a": 0, "b": 0, "expected_outputs": {"sum": 0, "carry": 0}}
a carry record : {"a": 15, "b": 15, "expected_outputs": {"sum": 14, "carry": 1}}
value types    : {'a': 'int', 'b': 'int', 'expected_outputs': 'dict'} | expected_outputs types: {'sum': 'int', 'carry': 'int'}


**Contents of the pattern file:** a JSON list with one record per test pattern. Each record has

* one key per **DUT input** (`a`, `b`) holding the stimulus as a decimal integer (the LLM emits binary strings; the pipeline converts them, which is why `"0101"` shows up here as `5`), and
* an **`expected_outputs`** object mapping each **DUT output** (`sum`, `carry`) to the integer the golden model returned for that stimulus.

The file is the bridge between steps 4 and 5: `GoldenModelGenerator.compute_golden_outputs` creates it by running the model over every pattern, and `TestbenchUpdater` reads it to place the expected constants into the testbench. If the model raised an exception for a pattern the record would instead carry `expected_outputs: null` plus an `error` string (none here).

### 5.3 Updater effect — before/after (`testbench_initial.v` → `testbench_final.v`)

In [22]:
diff_initial_final("adder", context=1, max_lines=48)

--- testbench_initial.v
+++ testbench_final.v
@@ -11,2 +11,7 @@
     initial begin
+        integer passed_tests;
+        integer failed_tests;
+        passed_tests = 0;
+        failed_tests = 0;
+
         // Test 1 (corner)
@@ -14,2 +19,17 @@
         $display("Test 1: a=%b b=%b", a, b);
+        #10; // let outputs settle
+        if (sum === 4'd0) begin
+            $display("  ✓ Test 1: sum = %0d (expected: 0)", sum);
+            passed_tests = passed_tests + 1;
+        end else begin
+            $display("  ✗ Test 1: sum = %0d (expected: 0)", sum);
+            failed_tests = failed_tests + 1;
+        end
+        if (carry === 1'd0) begin
+            $display("  ✓ Test 1: carry = %0d (expected: 0)", carry);
+            passed_tests = passed_tests + 1;
+        end else begin
+            $display("  ✗ Test 1: carry = %0d (expected: 0)", carry);
+            failed_tests = failed_tests + 1;
+        end
 
@@ -18,2 +38,17 @@
         $display("Test 2: a=%b b=%b", a, b);
+

**Concrete before/after difference.** `testbench_initial.v` only *applies and prints* stimuli — for each test it sets `a`/`b` and calls `$display("Test N: a=%b b=%b", ...)`; nothing is ever checked, so it would "pass" with any DUT. The updater turned it into a self-checking testbench by adding, per test, (1) a `#10;` delay so the combinational outputs settle, (2) an `if (sum === 4'dX)` / `if (carry === 1'dX)` compare against the golden constant with `✓`/`✗` messages that print actual and expected values, and (3) `passed_tests` / `failed_tests` counters; at the end of the file it appended the summary block with `ALL TESTS PASSED` / `SOME TESTS FAILED`. Using `===` (case equality) also means an `x`/`z` output counts as a failure rather than slipping through.

## 6. Part III — Custom module: `sat_adder8` (8-bit unsigned saturating adder with carry-in)

**Why this block:** same complexity class as the tutorial adder, but its behaviour has a precise boundary (exactly 255 vs. 256) and the carry-in can be the only thing that pushes the sum over the edge. Those are the cases where hand-written or naive LLM tests tend to be thin — and where an off-by-one bug hides.

**DUT interface**

| Port | Dir | Width | Meaning |
|---|---|---|---|
| `a` | in | 8 | unsigned operand |
| `b` | in | 8 | unsigned operand |
| `cin` | in | 1 | carry-in, added as 0/1 |
| `sum` | out | 8 | saturated result |
| `sat` | out | 1 | 1 iff the true sum exceeded 255 |

In [23]:
sat_description = """
An 8-bit unsigned saturating adder with carry-in. The Verilog module is named sat_adder8.

Ports:
- Input 'a': 8-bit unsigned operand (range 0..255)
- Input 'b': 8-bit unsigned operand (range 0..255)
- Input 'cin': 1-bit carry-in, added to the sum as the value 0 or 1
- Output 'sum': 8-bit unsigned saturated result
- Output 'sat': 1-bit saturation flag

Behaviour (purely combinational, no clock, no state):
- Let total = a + b + cin computed at full precision (no wraparound), so total is in 0..511.
- If total <= 255: sum = total and sat = 0.
- If total > 255: sum = 255 (8'hFF) and sat = 1. The result never wraps around to a small value.

Corner cases that must be handled exactly:
- total == 255 is NOT saturation: sum = 255 and sat = 0 (for example 254+1+0, 254+0+1, 255+0+0).
- total == 256 is the first saturating case: sum = 255 and sat = 1 (for example 255+0+1, 254+1+1, 128+128+0).
- The carry-in alone can cause saturation (255+0+1) or can leave the result at exactly 255 (254+0+1).
- 0+0+0 gives sum = 0 and sat = 0; 255+255+1 gives sum = 255 and sat = 1.

Verification requirements for the generated tests: at least 20 patterns in total, including directed corner
cases (zero operands, 255 operands, the 254/255/256 boundary with and without cin) and randomized patterns.
"""

sat_verilog_code = """
module sat_adder8 (
    input wire [7:0] a,
    input wire [7:0] b,
    input wire cin,
    output wire [7:0] sum,
    output wire sat
);
    // 9-bit full-precision result: bit 8 is set only when the true sum exceeds 255
    wire [8:0] full;
    assign full = a + b + cin;
    assign sat  = full[8];
    assign sum  = full[8] ? 8'hFF : full[7:0];
endmodule
"""
write("custom/sat_adder8.v", sat_verilog_code)
print(sat_description); print(sat_verilog_code)


An 8-bit unsigned saturating adder with carry-in. The Verilog module is named sat_adder8.

Ports:
- Input 'a': 8-bit unsigned operand (range 0..255)
- Input 'b': 8-bit unsigned operand (range 0..255)
- Input 'cin': 1-bit carry-in, added to the sum as the value 0 or 1
- Output 'sum': 8-bit unsigned saturated result
- Output 'sat': 1-bit saturation flag

Behaviour (purely combinational, no clock, no state):
- Let total = a + b + cin computed at full precision (no wraparound), so total is in 0..511.
- If total <= 255: sum = total and sat = 0.
- If total > 255: sum = 255 (8'hFF) and sat = 1. The result never wraps around to a small value.

Corner cases that must be handled exactly:
- total == 255 is NOT saturation: sum = 255 and sat = 0 (for example 254+1+0, 254+0+1, 255+0+0).
- total == 256 is the first saturating case: sum = 255 and sat = 1 (for example 255+0+1, 254+1+1, 128+128+0).
- The carry-in alone can cause saturation (255+0+1) or can leave the result at exactly 255 (254+0+1).
- 0

### 6.1 Run the pipeline

In [24]:
print("=" * 80, f"\nPipeline run: sat_adder8 | LLM mode = {LLM_MODE} | model = {LLM_MODEL}\n", "=" * 80, sep="")
sat_result = make_pipeline().run(description=sat_description, verilog_code=sat_verilog_code, output_dir="custom")
list_artifacts("custom")

Pipeline run: sat_adder8 | LLM mode = replay | model = claude-sonnet-5-5 (recorded responses in llm_responses/)
LLM-Aided Testbench Generation Pipeline

[Step 1-2] Input: Natural language description and Verilog code received
Description length: 1288 characters
Verilog code length: 359 characters

[Step 3] Generating testbench with test patterns...
    [replay] llm_responses/sat_adder8/01_testbench_response.txt  (8,710 chars)
  - Generated testbench with 40 test patterns
  - Module: sat_adder8
  - Saved initial testbench to: custom/testbench_initial.v

[Step 4] Generating Python golden model and computing expected outputs...
    [replay] llm_responses/sat_adder8/02_golden_response.txt  (350 chars)
  - Generated Python golden model (335 characters)
  - Saved Python golden model to: custom/golden_model.py
  - Computing golden outputs for all test patterns...
  - Successfully computed outputs for 40/40 patterns
  - Saved test patterns with golden outputs to: custom/test_patterns_with_gold

### 6.2 Quality of tests — are there corner cases *and* random patterns (>= 20)?

In [25]:
pats = json.load(open("custom/test_patterns_with_golden.json"))
tot = lambda p: p["a"] + p["b"] + p["cin"]
is_corner = lambda p: (p["a"] in (0, 255) or p["b"] in (0, 255) or tot(p) in (254, 255, 256, 255 + 255 + 1))
corner = [p for p in pats if is_corner(p)]
other = [p for p in pats if not is_corner(p)]
cov = {
    "total patterns":                         len(pats),
    "directed corner (0/255 operand or 254-256 boundary)": len(corner),
    "other / randomized-looking":             len(other),
    "sat == 1 cases":                         sum(p["expected_outputs"]["sat"] == 1 for p in pats),
    "sat == 0 cases":                         sum(p["expected_outputs"]["sat"] == 0 for p in pats),
    "total == 255 exactly (must NOT saturate)": sum(tot(p) == 255 for p in pats),
    "total == 256 exactly (first saturating)":  sum(tot(p) == 256 for p in pats),
    "cin = 1 cases":                          sum(p["cin"] == 1 for p in pats),
    "cin alone decides saturation (a+b==255, cin=1)": sum(p["a"] + p["b"] == 255 and p["cin"] == 1 for p in pats),
}
for k, v in cov.items(): print(f"  {v:4d}  {k}")
tbl = open("custom/testbench_initial.v", encoding="utf-8").read()
print("\nLabels the LLM put in testbench_initial.v comments:",
      {k: len(re.findall(rf"\(\s*{k}\s*\)", tbl)) for k in ("corner", "random")})
print("\nChecklist:",
      "[x]" if len(pats) >= 20 else "[ ]", ">= 20 patterns |",
      "[x]" if len(corner) and len(other) else "[ ]", "corner + random mix |",
      "[x]" if cov["total == 255 exactly (must NOT saturate)"] and cov["total == 256 exactly (first saturating)"] else "[ ]", "255/256 boundary covered")

    40  total patterns
    22  directed corner (0/255 operand or 254-256 boundary)
    18  other / randomized-looking
    20  sat == 1 cases
    20  sat == 0 cases
    10  total == 255 exactly (must NOT saturate)
     7  total == 256 exactly (first saturating)
    19  cin = 1 cases
     5  cin alone decides saturation (a+b==255, cin=1)

Labels the LLM put in testbench_initial.v comments: {'corner': 24, 'random': 16}

Checklist: [x] >= 20 patterns | [x] corner + random mix | [x] 255/256 boundary covered


### 6.3 Generated artifacts and checking logic

In [26]:
show_file("custom/golden_model.py")

--- custom/golden_model.py (9 lines shown) ------------------------------
   1: def sat_adder8_golden(a, b, cin):
   2:     """Reference model of an 8-bit unsigned saturating adder with carry-in.
   3: 
   4:     sum = min(a + b + cin, 255); sat = 1 only if the true sum exceeds 255.
   5:     """
   6:     total = (a & 0xFF) + (b & 0xFF) + (cin & 1)
   7:     if total > 255:
   8:         return {'sum': 255, 'sat': 1}
   9:     return {'sum': total, 'sat': 0}


In [27]:
print(json.dumps(pats[:2], indent=1), "\n  ... (%d records total)" % len(pats))

[
 {
  "a": 0,
  "b": 0,
  "cin": 0,
  "expected_outputs": {
   "sum": 0,
   "sat": 0
  }
 },
 {
  "a": 0,
  "b": 0,
  "cin": 1,
  "expected_outputs": {
   "sum": 1,
   "sat": 0
  }
 }
] 
  ... (40 records total)


In [28]:
show_checking_sections("custom/testbench_final.v", tests_shown=2)

--- custom/testbench_final.v: declarations + counters + first 2 tests ---
   1: `timescale 1ns/1ps
   2: 
   3: module tb_sat_adder8;
   4:     reg [7:0] a;
   5:     reg [7:0] b;
   6:     reg cin;
   7:     wire [7:0] sum;
   8:     wire sat;
   9: 
  10:     sat_adder8 uut (.a(a), .b(b), .cin(cin), .sum(sum), .sat(sat));
  11: 
  12:     initial begin
  13:         integer passed_tests;
  14:         integer failed_tests;
  15:         passed_tests = 0;
  16:         failed_tests = 0;
  17: 
  18:         // Test 1 (corner)
  19:         a = 8'b00000000; b = 8'b00000000; cin = 1'b0;
  20:         $display("Test 1: a=%b b=%b cin=%b", a, b, cin);
  21:         #10; // let outputs settle
  22:         if (sum === 8'd0) begin
  23:             $display("  ✓ Test 1: sum = %0d (expected: 0)", sum);
  24:             passed_tests = passed_tests + 1;
  25:         end else begin
  26:             $display("  ✗ Test 1: sum = %0d (expected: 0)", sum);
  27:             failed_tests = failed_t

### 6.4 Final verification (passing run)

In [29]:
sat_sim = run_sim("custom/sat_adder8.v", "custom/testbench_final.v", "custom", tail=12)
assert sat_sim["ok"] and sat_sim["failed"] == 0, "custom testbench did not pass"
print(f"\n==> sat_adder8 PASS: {sat_sim['passed']} checks passed, {sat_sim['failed']} failed")

$ iverilog -g2012 -o custom/sim.vvp custom/sat_adder8.v custom/testbench_final.v
(compiled with no errors or warnings)
$ vvp custom/sim.vvp
   ... (117 lines trimmed; full log saved) ...
Test 40: a=11110101 b=00001100 cin=0
  ✓ Test 40: sum = 255 (expected: 255)
  ✓ Test 40: sat = 1 (expected: 1)

========== Test Summary ==========
Total Checks: 80
Passed: 80
Failed: 0
ALL TESTS PASSED

custom/testbench_final.v:786: $finish called at 400000 (1ps)

==> sat_adder8 PASS: 80 checks passed, 0 failed


## 7. Part IV — Bug detection on `sat_adder8`

**Intentional bug:** the saturation flag is computed with `>=` instead of `>`, i.e. `sat = (full >= 255)`. That is a classic off-by-one: the sum value is still right, only `sat` is wrong, and **only when the true total is exactly 255** — 511 of the 131,072 possible `(a, b, cin)` combinations (about 0.4%). A handful of random vectors will usually miss it (16 random patterns hit it with roughly 6% probability); the directed boundary patterns generated in Part III should catch it.

In [30]:
fixed_src = open("custom/sat_adder8.v", encoding="utf-8").read()
bug_src = fixed_src.replace("assign sat  = full[8];", "assign sat  = (full >= 9'd255);   // BUG: should be full[8] (i.e. > 255)")
assert bug_src != fixed_src
os.makedirs("custom/bug_demo", exist_ok=True)
write("custom/bug_demo/sat_adder8_buggy.v", bug_src)
print("Diff: fixed DUT -> buggy DUT")
print("\n".join(difflib.unified_diff(fixed_src.splitlines(), bug_src.splitlines(), "sat_adder8.v (fixed)", "sat_adder8_buggy.v", lineterm="", n=0)))

Diff: fixed DUT -> buggy DUT
--- sat_adder8.v (fixed)
+++ sat_adder8_buggy.v
@@ -12 +12 @@
-    assign sat  = full[8];
+    assign sat  = (full >= 9'd255);   // BUG: should be full[8] (i.e. > 255)


### 7.1 Re-run `testbench_final.v` against the buggy DUT (expect FAIL)

In [31]:
bug_run = run_sim("custom/bug_demo/sat_adder8_buggy.v", "custom/testbench_final.v", "custom/bug_demo",
                  tag="buggy", log="simulation_fail.log", show=False)
print("$ iverilog -g2012 -o custom/bug_demo/buggy.vvp custom/bug_demo/sat_adder8_buggy.v custom/testbench_final.v")
print("$ vvp custom/bug_demo/buggy.vvp\n")
out = bug_run["stdout"].splitlines()
# show every failing check with the test-case line that produced it, then the summary
for i, l in enumerate(out):
    if "✗" in l:
        tn = re.search(r"Test (\d+)", l).group(1)
        stim = next(x for x in out if x.startswith(f"Test {tn}:"))
        print(stim); print(l)
print("\n".join(out[next(i for i, l in enumerate(out) if "Test Summary" in l) - 1:]))
print(f"\n==> BUG DETECTED: {bug_run['failed']} failing check(s) out of {bug_run['passed'] + bug_run['failed']}")
assert bug_run["failed"] and bug_run["failed"] > 0

$ iverilog -g2012 -o custom/bug_demo/buggy.vvp custom/bug_demo/sat_adder8_buggy.v custom/testbench_final.v
$ vvp custom/bug_demo/buggy.vvp

Test 3: a=11111111 b=00000000 cin=0
  ✗ Test 3: sat = 1 (expected: 0)
Test 4: a=00000000 b=11111111 cin=0
  ✗ Test 4: sat = 1 (expected: 0)
Test 9: a=11111110 b=00000001 cin=0
  ✗ Test 9: sat = 1 (expected: 0)
Test 10: a=00000001 b=11111110 cin=0
  ✗ Test 10: sat = 1 (expected: 0)
Test 11: a=11111110 b=00000000 cin=1
  ✗ Test 11: sat = 1 (expected: 0)
Test 12: a=11111101 b=00000001 cin=1
  ✗ Test 12: sat = 1 (expected: 0)
Test 15: a=10000000 b=01111111 cin=0
  ✗ Test 15: sat = 1 (expected: 0)
Test 18: a=01111111 b=10000000 cin=0
  ✗ Test 18: sat = 1 (expected: 0)
Test 21: a=10101010 b=01010101 cin=0
  ✗ Test 21: sat = 1 (expected: 0)
Test 24: a=11001000 b=00110111 cin=0
  ✗ Test 24: sat = 1 (expected: 0)

========== Test Summary ==========
Total Checks: 80
Passed: 70
Failed: 10
SOME TESTS FAILED

custom/testbench_final.v:786: $finish called at 4000

### 7.2 Fix the DUT and re-run (expect PASS)

In [32]:
write("custom/sat_adder8.v", fixed_src)          # restore the fixed source (this is what ships)
fix_run = run_sim("custom/sat_adder8.v", "custom/testbench_final.v", "custom/bug_demo",
                  tag="fixed", log="simulation_pass_after_fix.log", tail=10)
assert fix_run["ok"] and fix_run["failed"] == 0
print(f"\n==> after fix: {fix_run['passed']} passed, {fix_run['failed']} failed")

$ iverilog -g2012 -o custom/bug_demo/fixed.vvp custom/sat_adder8.v custom/testbench_final.v
(compiled with no errors or warnings)
$ vvp custom/bug_demo/fixed.vvp
   ... (119 lines trimmed; full log saved) ...
  ✓ Test 40: sat = 1 (expected: 1)

========== Test Summary ==========
Total Checks: 80
Passed: 80
Failed: 0
ALL TESTS PASSED

custom/testbench_final.v:786: $finish called at 400000 (1ps)

==> after fix: 80 passed, 0 failed


### 7.3 Extra: mutation sweep — does the same testbench catch other plausible bugs?

In [33]:
mutants = {
  "M1 sat uses >= 255 (off-by-one, the demo bug)": ("assign sat  = full[8];", "assign sat  = (full >= 9'd255);"),
  "M2 saturates to 8'hFE instead of 8'hFF":        ("8'hFF", "8'hFE"),
  "M3 carry-in ignored":                           ("assign full = a + b + cin;", "assign full = a + b;"),
  "M4 sat flag ignores cin (sum correct)":         ("assign sat  = full[8];", "assign sat  = ((a + b) > 9'd255);"),
  "M5 operand b replaced by a":                    ("assign full = a + b + cin;", "assign full = a + a + cin;"),
}
print(f"{'mutant':50s} {'failed checks':>13s}  first failing test")
for name, (old, new) in mutants.items():
    assert old in fixed_src, name
    tag = name.split()[0].lower()
    write(f"custom/bug_demo/sat_adder8_{tag}.v", fixed_src.replace(old, new))
    r = run_sim(f"custom/bug_demo/sat_adder8_{tag}.v", "custom/testbench_final.v", "custom/bug_demo",
                tag=tag, log=None, show=False)
    first = re.search(r"✗ Test (\d+)", r["stdout"])
    print(f"{name:50s} {r['failed']:>13d}  {('Test ' + first.group(1)) if first else '-'}   {'DETECTED' if r['failed'] else 'SURVIVED'}")

mutant                                             failed checks  first failing test


M1 sat uses >= 255 (off-by-one, the demo bug)                 10  Test 3   DETECTED


M2 saturates to 8'hFE instead of 8'hFF                        20  Test 5   DETECTED


M3 carry-in ignored                                           13  Test 2   DETECTED


M4 sat flag ignores cin (sum correct)                          5  Test 5   DETECTED


M5 operand b replaced by a                                    28  Test 3   DETECTED


## 8. Final checklist against the assignment

In [34]:
required = ["testbench_initial.v", "golden_model.py", "test_patterns_with_golden.json", "testbench_final.v", "simulation_output.log"]
ok = True
for d in ("mux", "adder", "custom"):
    miss = [f for f in required if not os.path.exists(f"{d}/{f}")]
    ok &= not miss
    print(f"{d:7s}", "all artifacts present" if not miss else f"MISSING {miss}")
for name, s in (("mux2to1", mux_sim), ("adder4bit", adder_sim), ("sat_adder8", sat_sim)):
    print(f"  {name:11s} final testbench: {s['passed']:3d} passed / {s['failed']} failed")
print("  bug demo     : fail run ->", bug_run["failed"], "failing checks; after fix ->", fix_run["failed"], "failing")
nb_text = open("testbenchgen_hw.ipynb", encoding="utf-8").read() if os.path.exists("testbenchgen_hw.ipynb") else ""
leaked = re.findall(r"sk-[A-Za-z0-9_-]{20,}", nb_text)
print("  API key in notebook source:", "NONE" if not leaked else leaked)
print("\nTestbenchGen_Report.pdf is submitted alongside this notebook." )

mux     all artifacts present
adder   all artifacts present
custom  all artifacts present
  mux2to1     final testbench:   8 passed / 0 failed
  adder4bit   final testbench:  64 passed / 0 failed
  sat_adder8  final testbench:  80 passed / 0 failed
  bug demo     : fail run -> 10 failing checks; after fix -> 0 failing
  API key in notebook source: NONE

TestbenchGen_Report.pdf is submitted alongside this notebook.
